# Two-face lens driven by an idealized acoustic array
## Saved equilibrium, freshly checked optics, and acoustic wavefronts

This notebook presents the **post-synthesis** result: fixed complex source commands produce an acoustic field, whose cycle-mean traction balances gravity and capillarity on both interfaces. It does not prescribe the target pressure in place of the emitters.

The saved design-grid result is **0.513 µm maximum geometric spot radius**, with **20.2 / 24.4 nm** front/back pupil errors. Approximately 0.5 µm is acceptable for the spot; the separate 10 nm height and full-ray-coverage requirements are not met. Fixed-command refinement remains unfinished.

**What moves in the movie:** the real harmonic pressure field and boundary-source excitation over one acoustic period. Both cycle-mean surfaces remain stationary. This is not a switch-on or formation trajectory.

Idealized source models are the intended research scope; hardware calibration is not an acceptance requirement here. The implemented acoustics is longitudinal and lossless in the bulk, with passive Robin boundary absorption. No thermoviscous bulk dynamics, mean streaming, thermal feedback, curing, diffraction or 3D stability result is claimed.


In [ ]:
%matplotlib inline
from pathlib import Path
import hashlib, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import HTML, display
from acoustic_freeform.dual.config import DualConfig
from acoustic_freeform.dual.surface import DualSurface, CartesianPatch
from acoustic_freeform.dual.optics import trace_pair
from acoustic_freeform.dual.campaign import maximum_error
from acoustic_freeform.dual.acoustics import DualAcoustics

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'pyproject.toml').exists())
BASE = ROOT/'artifacts/noa61-emitter-2026-09-23'
RUN = BASE/'verify-448-c4-clear-spot/design104'
SYNTH = BASE/'force-448-c4-clear-spot'
read = lambda p: json.loads(p.read_text())
cfg = DualConfig(**read(RUN/'config.json'))
settings = read(SYNTH/'config.json')
material = read(ROOT/settings['material_reference'])
report = read(RUN/'validation.json')
regions = read(RUN/'source-regions.json')
state = np.load(RUN/'state.npz')
q, drive = state['coefficients_m'], state['source_velocity_m_s']
space = DualSurface(cfg)
targets = [CartesianPatch(cfg, j, **face) for j, face in enumerate(settings['case']['faces'])]
assert report['coupled_root_converged']
assert np.array_equal(drive, np.load(SYNTH/'best-source.npz')['source_velocity_m_s'])
assert len(drive) == len(regions['regions']) == 448
assert cfg.gravity_m_s2 == 9.81
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
colors = ['#1479b8', '#e47d24']
print('Input state SHA256:', hashlib.sha256((RUN/'state.npz').read_bytes()).hexdigest())
print('Source condition:', regions['source_condition'])
print('Frequency:', cfg.frequency_hz/1e6, 'MHz; active regions:', np.count_nonzero(abs(drive)>1e-14))


## 1. Apparatus and prescription

The cylinder has radius 4 mm and height 6 mm. Two pinned fluid interfaces separate water / uncured NOA 61 / water. Each optical pupil has radius 2 mm; outside it, the declared C4 annulus joins the wall. Both Cartesian surfaces share the intermediate laboratory conjugate.

The source model is **V·n = YP + g**, with peak phasors using exp(−iωt). The optimized command is g, not the total boundary velocity. There are 128 annuli per end cap and 64 sidewall bands per fluid layer. Central end-cap regions have zero active command; their passive boundary condition remains in the acoustic model.

All calculations and saved arrays use SI. Plot conversions are labelled. The 3D view uses equal geometric scale, with no surface exaggeration. Its source regions are annuli and bands—not individual point speakers.


In [ ]:
print('Laboratory conjugates z0, z1, z2 [mm]:', np.array(material['stigmatic_z_m'])*1e3)
print('Refractive indices:', material['indices'])
print('Density [kg/m³]:', cfg.density_kg_m3)
print('Sound speed [m/s]:', cfg.sound_speed_m_s)
print('Interfacial tension [N/m]:', cfg.surface_tension_n_m)
print('Gravity [m/s²]:', cfg.gravity_m_s2)
print('Assumptions: resin sound speed and resin–water interfacial tension are engineering inputs.')
print('Viscosities are not used by this stationary acoustic/mechanical solve.')

fig = plt.figure(figsize=(12,6), constrained_layout=True)
ax = fig.add_subplot(121, projection='3d')
theta = np.linspace(0, 2*np.pi, 49)
polygons, amplitudes = [], []
for reg in regions['regions']:
    j = reg['channel']
    if abs(drive[j]) < 1e-14:
        continue
    for a, b in zip(theta[:-1], theta[1:]):
        if reg['boundary'] == 'side':
            r0 = cfg.radius_m
            vertices = [(r0*np.cos(t), r0*np.sin(t), z)
                        for t,z in [(a,reg['z_min_m']),(b,reg['z_min_m']),
                                    (b,reg['z_max_m']),(a,reg['z_max_m'])]]
        else:
            z = reg['z_min_m']
            vertices = [(r*np.cos(t), r*np.sin(t), z)
                        for r,t in [(reg['r_min_m'],a),(reg['r_max_m'],a),
                                    (reg['r_max_m'],b),(reg['r_min_m'],b)]]
        polygons.append(np.array(vertices)*1e3)
        amplitudes.append(abs(drive[j]))
collection = Poly3DCollection(polygons, alpha=.20, linewidth=0, cmap='viridis')
collection.set_array(np.array(amplitudes)); collection.set_clim(0,1)
ax.add_collection3d(collection)
r3 = np.linspace(0,cfg.radius_m,30)
X,Y = r3[:,None]*np.cos(theta)*1e3, r3[:,None]*np.sin(theta)*1e3
for j in range(2):
    z = (cfg.levels_m[j+1]+space.evaluate(q[j],r3))*1e3
    ax.plot_wireframe(X,Y,np.broadcast_to(z[:,None],X.shape),color=colors[j],linewidth=.5)
ax.set(xlim=(-4,4),ylim=(-4,4),zlim=(-3,3),xlabel='x [mm]',ylabel='y [mm]',
       zlabel='z [mm]',title='Declared source regions and both mean faces')
ax.set_box_aspect((8,8,6)); ax.view_init(22,-55)
fig.colorbar(collection,ax=ax,shrink=.55,label='Peak command |g| [m/s]')
axes = [fig.add_subplot(222),fig.add_subplot(224)]
axes[0].plot(abs(drive),lw=1); axes[0].set(ylabel='|g| [m/s]',title='Saved commands — no re-optimization')
active = abs(drive)>1e-14
axes[1].scatter(np.flatnonzero(active),np.angle(drive[active]),s=4)
axes[1].set(xlabel='Source channel',ylabel='Phase [rad]')
for axis in axes:
    for boundary in (128,256,320,384):
        axis.axvline(boundary,color='gray',lw=.5)
plt.show()


## 2. Both surfaces and the fixed-detector spot

The launch cone is fixed using the target front pupil, exactly as in the saved validation. Rays are refracted through the **computed** surfaces at the prescribed detector; there is no best-focus adjustment or pupil shrinkage. The 2D spot plot uses azimuthal placement of axisymmetric meridional rays, not a diffraction intensity image.

The notebook retraces all 4001 saved launch radii and checks the saved spot coordinates and transmission mask. Reported maxima apply to this numerical model and sampling; they are not continuous-aperture certificates.


In [ ]:
r = np.linspace(0,cfg.radius_m,1601)
rp = np.linspace(0,cfg.clear_radius_m,4001)
target_h = np.array([t.evaluate(r) for t in targets])
actual_h = np.array([space.evaluate(v,r) for v in q])
trace = trace_pair(space,q,material['indices'],material['stigmatic_z_m'][0],
    material['stigmatic_z_m'][2],launch_radius_m=rp,
    launch_height_m=cfg.levels_m[1]+targets[0].evaluate(rp))
np.testing.assert_array_equal(trace['transmitted'],state['transmitted'])
np.testing.assert_allclose(trace['spots_m'],state['spots_m'],atol=1e-12,rtol=1e-7,equal_nan=True)
errors = [maximum_error(space,v,t)['max_error_m'] for v,t in zip(q,targets)]
np.testing.assert_allclose(errors,report['mean_pupil_max_error_m'],rtol=1e-7,atol=1e-13)
lost = np.flatnonzero(~trace['transmitted'])
print(f"Maximum spot radius: {trace['max_radius_m']*1e6:.6f} µm (surviving rays)")
print('Front/back maximum pupil height error [nm]:',np.array(errors)*1e9)
print(f"Transmitted: {trace['transmitted'].sum()} / {len(rp)}; lost launch radii [mm]:",rp[lost]*1e3)
print('Joint height + spot + coverage gate:',report['sampled_joint_gate'])

fig,axes = plt.subplots(1,3,figsize=(15,4.5),constrained_layout=True)
signed = np.r_[-r[:0:-1],r]
for j in range(2):
    for h,style in [(actual_h[j],'-'),(target_h[j],'--')]:
        axes[0].plot(signed*1e3,(cfg.levels_m[j+1]+np.r_[h[:0:-1],h])*1e3,
                     style,color=colors[j],lw=1.2)
    e = (space.evaluate(q[j],rp)-targets[j].evaluate(rp))*1e9
    axes[1].plot(rp*1e3,e,color=colors[j],label=['Front / lower','Back / upper'][j])
axes[0].set(xlabel='Signed radius [mm]',ylabel='z [mm]',title='Solid: computed; dashed: target')
axes[0].axvspan(-2,2,color='gray',alpha=.08)
axes[1].axhspan(-10,10,color='green',alpha=.1)
axes[1].set(xlabel='Pupil radius [mm]',ylabel='Height minus target [nm]',title='Absolute height error, no piston removal')
axes[1].legend()
axes[2].scatter(*trace['spots_m'].T*1e6,s=2,alpha=.35)
axes[2].add_patch(plt.Circle((0,0),.5,fill=False,color='green',ls='--',label='0.5 µm guide'))
axes[2].set(xlim=(-.65,.65),ylim=(-.65,.65),aspect='equal',
            xlabel='Detector x [µm]',ylabel='Detector y [µm]',title=f"Fixed detector; {len(lost)} ray(s) fail coverage")
axes[2].legend()
plt.show()


## 3. Fresh acoustic solve at the saved equilibrium

This cell solves the harmonic field again using the saved geometry and **unchanged commands**. It does not re-optimize, prescribe the desired pressure, or run another equilibrium search. It may take several minutes.

The force check retains all generalized forces on both complete interfaces, including the non-optical annuli. A small equilibrium residual is not a stability or formation result. Pressure plots below show the actual harmonic pressure and mean traction; they are different quantities.


In [ ]:
wave = DualAcoustics(cfg,space,linear_solver='static_condensed')
response = wave.solve(q,drive)
diagnostics = response.diagnostics(np.ones(1,complex))
mechanical,K,_ = space.mechanics(q)
residual = response.force(np.ones(1,complex))-mechanical
defect_coeff = np.linalg.solve(K,residual).reshape(q.shape)
force_defect = max(space.polynomial_maximum(v)['max_abs_m'] for v in defect_coeff)
assert force_defect < 1e-12, 'Fresh full-force check failed'
np.testing.assert_allclose(diagnostics['source_power_w'],
    report['wave_diagnostics']['source_power_w'],rtol=1e-6)
print('Fresh full-force compliance defect [pm]:',force_defect*1e12)
print('Source acoustic power [W]:',diagnostics['source_power_w'])
print('Sampled peak acoustic pressure [MPa]:',diagnostics['sampled_pressure_peak_pa']/1e6)
print('Wave linear residual:',diagnostics['wave_linear_residual'])
print('Power closure relative error:',diagnostics['power_relative_error'])


### Mean load and gravity

For each interface the mechanical load is −σ div(∇h/√(1+|∇h|²)) + Δρ gh, up to a spatially constant volume multiplier. The plots align that constant using an area-weighted mean. This is a pressure gauge choice, not a spatial fit.

The lower interface has Δρ = −234 kg/m³ and the upper +234 kg/m³: gravity is explicitly retained with opposite signs. The load curves are quadrature-sampled diagnostics; the full variational residual above is the equilibrium test.


In [ ]:
fig,axes = plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
omega = 2*np.pi*cfg.frequency_hz
for j in range(2):
    rr = response.radial_m[j]
    rho0,rho1 = cfg.density_kg_m3[j:j+2]
    c0,c1 = cfg.sound_speed_m_s[j:j+2]
    p = response.pressure[j][:,0]
    vn = response.velocity[j][:,0]
    gt = response.tangent_gradient[j][:,0]
    traction = .25*(1/(rho0*c0*c0)-1/(rho1*c1*c1))*abs(p)**2
    traction += (rho0-rho1)/4*abs(vn)**2
    traction += (rho0-rho1)/(4*omega**2*rho0*rho1)*abs(gt)**2
    def load(evaluate):
        h,s,ss = [evaluate(rr,d) for d in (0,1,2)]
        curvature = ss/(1+s*s)**1.5 + s/(rr*np.sqrt(1+s*s))
        return -cfg.surface_tension_n_m[j]*curvature+(rho0-rho1)*cfg.gravity_m_s2*h
    actual_load = load(lambda r,d: space.evaluate(q[j],r,d))
    target_load = load(lambda r,d: targets[j].evaluate(r,d))
    weights = response.weights_m2[j]
    centered = lambda v: v-np.average(v,weights=weights)
    order = np.argsort(rr)
    for values,label,style in [(traction,'Acoustic traction','-'),
                               (actual_load,'Computed-surface mechanical load','--'),
                               (target_load,'Target mechanical load',':')]:
        axes[j,0].plot(rr[order]*1e3,centered(values)[order],style,label=label,lw=1)
    axes[j,0].set(xlabel='Radius [mm]',ylabel='Mean load, constant removed [Pa]',
                  title=['Front / lower','Back / upper'][j])
    axes[j,0].legend(fontsize=8)
    axes[j,1].plot(rr[order]*1e3,abs(p[order])/1e6,color=colors[j],lw=1)
    axes[j,1].set(xlabel='Radius [mm]',ylabel='Peak |P| [MPa]',title='Harmonic pressure at interface')
plt.show()


## 4. Embedded wavefront movie — one acoustic period

Colour shows **Re[P exp(−iωt)]**, using the freshly solved complex field. Boundary colours separately show **Re[g exp(−iωt)]**. Fixed colour scales prevent artificial amplification from frame to frame. The cavity field includes interference and reflections; this is not an animation of independent expanding rings or of sound switch-on.

Playback is slowed enormously; displayed time is physical, in nanoseconds. The 2D meridional view is sufficient for the axisymmetric field. The two cycle-mean surfaces are stationary black curves. No optical spot evolution is implied by this carrier movie.

For rendering only, pressure is sampled at acoustic mesh vertices and linearly coloured across triangles; the solve uses fourth-order acoustic elements. The movie is qualitative visualization, not a replacement for the high-order force calculation.


In [ ]:
xy = response.basis.mapping.geometry(wave.flat.p)[0]*cfg.radius_m
pressure = response.solution[response.basis.nodal_dofs[0],0]
tri = mtri.Triangulation(xy[0]*1e3,xy[1]*1e3,wave.flat.t.T)
segments = []
for reg in regions['regions']:
    segments.append([(reg['r_min_m']*1e3,reg['z_min_m']*1e3),
                     (reg['r_max_m']*1e3,reg['z_max_m']*1e3)])
fig,ax = plt.subplots(figsize=(7,7),constrained_layout=True)
limit = float(np.max(abs(pressure)))/1e6
field = ax.tripcolor(tri,pressure.real/1e6,shading='gouraud',
                     cmap='RdBu_r',vmin=-limit,vmax=limit,rasterized=True)
fig.colorbar(field,ax=ax,label='Instantaneous acoustic pressure [MPa]',shrink=.75)
boundary = LineCollection(segments,cmap='PiYG',linewidths=4)
boundary.set_clim(-1,1); boundary.set_array(drive.real)
ax.add_collection(boundary)
fig.colorbar(boundary,ax=ax,label='Instantaneous source command g [m/s]',shrink=.75,pad=.15)
for j in range(2):
    ax.plot(r*1e3,(cfg.levels_m[j+1]+actual_h[j])*1e3,color='black',lw=1)
ax.set(xlim=(-.08,4.08),ylim=(-3.08,3.08),aspect='equal',
       xlabel='Radius [mm]',ylabel='z [mm]')
title = ax.set_title('')
times = np.arange(32)/(32*cfg.frequency_hz)
def update(k):
    phase = np.exp(-1j*omega*times[k])
    field.set_array((pressure*phase).real/1e6)
    boundary.set_array((drive*phase).real)
    title.set_text(f'Carrier wavefronts — physical time {times[k]*1e9:.2f} ns\nStationary mean lens; not formation')
    return field,boundary,title
assert np.max(abs(pressure.real-(pressure*np.exp(-1j*np.pi/2)).real)) > 0
update(0)
animation = FuncAnimation(fig,update,frames=len(times),interval=110,blit=False)
embedded = animation.to_jshtml(embed_frames=True,default_mode='loop')
plt.close(fig)
display(HTML(embedded))


## 5. What this notebook verifies

- The saved source commands, both surface profiles and the ray trace belong to the same post-synthesis state.
- A fresh acoustic solve reproduces the saved power and balances all discrete mechanical forces with gravity included.
- The embedded wavefronts come from that solved pressure field, not a target interpolation.
- The 0.513 µm spot is a **maximum over surviving sampled rays**, not an RMS radius or a whole-pupil pass.
- Both height errors exceed 10 nm; the edge-ray coverage check fails. The interrupted surface128 run is not convergence evidence.
- Formation, stability and spot evolution in physical formation time require a subsequent fluid trajectory with these same commands. Newton iterations are not physical time.

### Provenance and further reading

[Campaign index](../artifacts/noa61-emitter-2026-09-23/README.md) ·
[Saved validation](../artifacts/noa61-emitter-2026-09-23/verify-448-c4-clear-spot/design104/validation.json) ·
[Source commands](../artifacts/noa61-emitter-2026-09-23/verify-448-c4-clear-spot/design104/source-commands.csv) ·
[Material assumptions](../docs/resin-design-reference.md) ·
[Canonical theory](../artifacts/theory/acoustic-fluid-shaping-theory.pdf)

Notebook 08 is a separate prescribed-load formation study and is not the dynamics of this source array.


In [ ]:
checks = {
    'saved_coupled_root_converged': report['coupled_root_converged'],
    'fresh_force_compliance_defect_m': float(force_defect),
    'fresh_source_power_w': float(diagnostics['source_power_w']),
    'max_geometric_spot_radius_m': float(trace['max_radius_m']),
    'mean_pupil_max_error_m': errors,
    'transmitted_count': int(trace['transmitted'].sum()),
    'launch_count': len(rp),
    'all_rays_transmitted': bool(trace['all_rays_transmitted']),
    'embedded_carrier_frames': len(times),
    'acoustic_period_s': 1/cfg.frequency_hz,
    'formation_trajectory': False,
    'fixed_command_refinement_complete': False,
}
print(json.dumps(checks,indent=2))
